In [ ]:
!pip install --upgrade pip
!pip install --upgrade torch transformers

In [ ]:
import tensorflow as tf
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

In [ ]:
def start_final_chatbot():
    model_name = "Qwen/Qwen2.5-1.5B-Instruct"

    print(f"Loading {model_name} on {device.upper()}...")

    # Load tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype="auto",
        device_map="auto"
    )

    # 3. Chat History Setup
    messages = [
        {"role": "system", "content": "You are a helpful and concise AI assistant."}
    ]

    print("\n" + "="*40)
    print("AI ASSISTANT: Ready! (Type 'exit' to quit)")
    print("="*40 + "\n")

    while True:
        # 4. User Input
        user_input = input("User: ")

        if user_input.lower() in ['exit', 'quit']:
            print("Chatbot: Goodbye!")
            break

        # Add user message to history
        messages.append({"role": "user", "content": user_input})

        # 5. Fast Generation Logic
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

        with torch.no_grad():
            generated_ids = model.generate(
                **model_inputs,
                max_new_tokens=256,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id
            )

        # Remove input tokens from the output to get only the new response
        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]

        response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

        #Display and Store Response
        print(f"Chatbot: {response}")
        messages.append({"role": "assistant", "content": response})

if __name__ == "__main__":
    start_final_chatbot()